In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    when,
    trim,
    upper,
    length,
    regexp_replace
)

SOURCE_FILE = "ecommerce_enderecos.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}

EXPECTED_COLUMNS = [
    "id_endereco",
    "id_cliente",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "cidade",
    "estado",
    "latitude",
    "longitude",
    "apelido",
    "is_principal"
]

adls_options = get_adls_options()

print(f"Arquivo analisado: {SOURCE_FILE}")
print(f"Caminho RAW: {SOURCE_PATH}")

In [0]:
df_raw = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

actual_columns = df_raw.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

print("Leitura da RAW concluída.")
print(f"Total de linhas: {df_raw.count()}")
print(f"Total de colunas: {len(actual_columns)}")
print(f"Colunas ausentes: {missing_columns}")
print(f"Colunas extras: {extra_columns}")

df_raw.printSchema()

display(df_raw.limit(10))

In [0]:
total_linhas = df_raw.count()
ids_endereco_distintos = df_raw.select("id_endereco").distinct().count()
ids_cliente_distintos = df_raw.select("id_cliente").distinct().count()

print(f"Total de linhas: {total_linhas}")
print(f"IDs de endereço distintos: {ids_endereco_distintos}")
print(f"IDs de endereço duplicados: {total_linhas - ids_endereco_distintos}")
print(f"Clientes distintos com endereço: {ids_cliente_distintos}")

df_validacao_chaves = df_raw.select(
    count("*").alias("total_linhas"),
    spark_sum(when(col("id_endereco").isNull() | (trim(col("id_endereco")) == ""), 1).otherwise(0)).alias("id_endereco_nulo_ou_vazio"),
    spark_sum(when(col("id_cliente").isNull() | (trim(col("id_cliente")) == ""), 1).otherwise(0)).alias("id_cliente_nulo_ou_vazio")
)

display(df_validacao_chaves)

In [0]:
df_nulos_vazios = df_raw.select([
    spark_sum(
        when(col(c).isNull() | (trim(col(c)) == ""), 1).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
])

display(df_nulos_vazios)